In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error


Without Scaing

In [2]:
boston = pd.read_csv('../datasets/Boston.csv')
x,y = boston.drop('medv', axis = 1), boston['medv']
x_train, x_test, y_train, y_test = train_test_split(x,y, random_state=25, test_size=0.3)

sgd = SGDRegressor(random_state=25)
sgd.fit(x_train, y_train)
y_pred = sgd.predict(x_test)
mean_absolute_error(y_test,y_pred)

312844902785337.25

with scaling

In [3]:
boston = pd.read_csv('../datasets/Boston.csv')
x,y = boston.drop('medv', axis = 1), boston[['medv']]#here since we are scaling y also, so we have to make it as a df and not a series
x_train, x_test, y_train, y_test = train_test_split(x,y, random_state=25, test_size=0.3)
scl_x, scl_y = MinMaxScaler(), MinMaxScaler()
x_trn_scl, y_trn_scl = scl_x.fit_transform(x_train), scl_y.fit_transform(y_train)
x_tst_scl = scl_x.transform(x_test)

sgd = SGDRegressor(random_state=25)
sgd.fit(x_trn_scl, y_trn_scl)
y_pred_scl = sgd.predict(x_tst_scl)
y_pred = scl_y.inverse_transform(y_pred_scl.reshape(-1, 1))
mean_absolute_error(y_test,y_pred)

c:\Users\dbda.STUDENTSDC\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


4.309098492252893

In [8]:
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')


etas = [0.001, 0.01,0.1 ,0.2,0.4]
lr_sch = ['constant', 'optimal', 'adaptive', 'invscaling']
scores = []

for e in tqdm(etas):
    for lr in lr_sch:
        sgd = SGDRegressor(random_state=25, eta0=e, learning_rate=lr)
        sgd.fit(x_trn_scl, y_trn_scl)
        y_pred_scl = sgd.predict(x_tst_scl)
        y_pred = scl_y.inverse_transform(y_pred_scl.reshape(-1, 1))
        scores.append([e, lr, mean_absolute_error(y_test,y_pred)])

df_scores = pd.DataFrame(data = scores, columns=['Etas', 'Learing Rate(Scheduler)', 'score'] )
df_scores.sort_values('score', ascending=True)

100%|██████████| 5/5 [00:00<00:00, 52.50it/s]


,Etas,Learing Rate(Scheduler),score
10,0.100,adaptive,2.984731
18,0.400,adaptive,3.031574
14,0.200,adaptive,3.084227
6,0.010,adaptive,3.122314
11,0.100,invscaling,3.146949
4,0.010,constant,3.270725
15,0.200,invscaling,3.274445
19,0.400,invscaling,3.352337
8,0.100,constant,3.445734
9,0.100,optimal,3.536500
